In [ ]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
import pandas as pd

import OpenDartReader

pd.options.display.max_colwidth = None
pd.options.display.max_rows = None

def make_clickable(val):
    return f'<a href="{val}" target="_blank">{val}</a>'

api_key = '888e4aca3aba61a62811ba87e0fc054e2d9f6ea0'

dart = OpenDartReader(api_key)

dart.list('이뮤노백스바이오')



In [ ]:
df = dart.sub_docs('20250403000343')
df.style.format({'url': make_clickable})

In [ ]:
# 1. 컬럼 너비 제한 해제 (URL이 잘리지 않게 함)
# 2. 클릭 가능한 링크로 변환하는 함수

# 3. 결과 출력 시 스타일 적용
df = dart.sub_docs('20250411000394')
df.style.format({'url': make_clickable})

## 감사보고서 PDF 일괄 다운로드 기능\n
이 셀을 먼저 실행하여 함수를 정의하세요.

In [ ]:
import os

def download_audit_reports(corp_code, start_date):
    print(f"Searching for reports for {corp_code} since {start_date}...")
    # 1. 보고서 목록 조회 (F: 정기공시 - 감사보고서 포함)
    reports = dart.list(corp_code, start=start_date, kind='F')
    
    if reports is None or len(reports) == 0:
        print("No reports found.")
        return

    # '감사보고서' 또는 '연결감사보고서' 키워드가 포함된 보고서만 필터링
    audit_reports = reports[reports['report_nm'].str.contains('감사보고서')]
    
    if len(audit_reports) == 0:
        print("No audit reports found in the search results.")
        return
    
    print(f"Found {len(audit_reports)} audit report(s).")

    if not os.path.exists('PDF'):
        os.makedirs('PDF')
        print("Created 'PDF' directory.")
        
    downloaded_count = 0
    for _, row in audit_reports.iterrows():
        rcp_no = row['rcept_no']
        report_nm = row['report_nm']
        print(f"\nChecking attachments for: {report_nm} ({rcp_no})")
        
        try:
            files = dart.attach_files(rcp_no)
            if not files:
                print("  No attachments found.")
                continue
                
            for filename, url in files.items():
                if filename.lower().endswith('.pdf'):
                    # 파일명에 특수문자 제거 및 경로 설정
                    clean_filename = filename.replace('/', '_').replace('\\', '_')
                    save_path = os.path.join('PDF', clean_filename)
                    
                    print(f"  Downloading: {filename}")
                    dart.download(url, save_path)
                    downloaded_count += 1
        except Exception as e:
            print(f"  Error accessing attachments: {e}")
                
    print(f"\nFinished! Successfully downloaded {downloaded_count} PDF(s) to 'PDF/' folder.")

### 다운로드 실행\n
아래 셀을 실행하면 실제 다운로드가 시작됩니다.

In [ ]:
# 구다이글로벌('01753163')의 2024년 이후 모든 감사보고서 PDF 다운로드 실행
download_audit_reports('01753163', '2024-01-01')